In [1]:
import os
import imageio.v3 as iio
from PIL import Image
from pathlib import Path

# import argparse
# parser = argparse.ArgumentParser(description="hello")
# parser.add_argument('-i', '--inpath', type=str, default=str(os.getcwd()), help="I will glob all pngs and jpgs from this folder")
# parser.add_argument('-o', '--outpath', type=str, default=str(os.path.join(os.getcwd(), 'hist_evolution')), help='I will make an mp4 and a gif with this name')
# parser.add_argument('-f', '--fps', type=int, default=10, help='How many fps')
# parser.add_argument('--match', type=str, default=None, help='If you give me a match, I will glob all pngs/jpgs matching this string')
# args = parser.parse_args()

def custom_walk(directory):
    for entry in directory.iterdir():
        if entry.is_dir():
            yield from custom_walk(entry)
        else:
            yield entry

def collect_images(image_folder: Path, match=None):
    """Return sorted list of image file paths."""
    images = []
    for filepath in sorted(custom_walk(image_folder)):
        if match and match not in filepath.name:
            continue
        if filepath.suffix.lower() in (".png", ".jpg", ".jpeg"):
            # print(filepath)
            images.append(filepath)
    print(len(images))
    return images


def make_gif(image_paths, output_path="animation.gif", fps=10):
    if not image_paths:
        print("No images found!")
        return
    frames = [Image.fromarray(iio.imread(path)) for path in image_paths]
    frames[0].save(
        output_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=1000 // fps,
        loop=0
    )
    
    print(f"GIF saved to {output_path}")


def make_mp4(image_paths, output_path="animation.mp4", fps=10):
    if not image_paths:
        print("No images found!")
        return
        
    '''
    frames = [iio.imread(path) for path in image_paths]
    iio.imwrite(output_path, frames, fps=fps, codec='libx264')
    '''
    
    with iio.imopen(output_path, "w", plugin="pyav") as writer:
        writer.init_video_stream("libx264", fps=fps)

        for path in image_paths:
            print(path)
            frame = iio.imread(path)[..., :3]
            h, w = frame.shape[:2]
            frame = frame[:h - (h%2),:w - (w%2)]
            writer.write_frame(frame)
    print(f"MP4 saved to {output_path}")

In [3]:
inpath = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/selected_events-data-1e20-crosser_muons_bwd"
outpath = "./gifs/"
fps=10
for match in [
            #   "muon-p_topology",
            #   "muon-dir_z_topology",
            #   "proton-p_topology",
            #   "proton-dir_z_topology",
            #   "tki-del_Tp_topology",
            #   "tki-del_p_topology",
            #   "tki-del_alpha_topology",
            #   "tki-del_phi_topology",
              "muon-dir_phi_topology"
]:

    folder = Path(inpath)
    image_list = collect_images(folder, match)

    make_gif(image_list, output_path=outpath + match + "_crosser_muons_bwd.gif", fps=fps)
    # make_mp4(image_list, output_path=outpath + ".mp4", fps=fps)

15
GIF saved to ./gifs/muon-dir_phi_topology_crosser_muons_bwd.gif
